# 03 PyQSAR3 Modeling

## Step 3.1: Native PyQSAR3 Pre-Filtering

This notebook restarts Phase 3 from the raw Mordred descriptor outputs and performs only pre-filtering. Native PyQSAR3 filtering tools are used for non-numeric, missing, infinite, and zero-variance descriptor cleanup. A train-set-only Pearson correlation pass is then applied because the installed PyQSAR3 package does not expose a dedicated high-correlation pre-filter function.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from pyqsar import data_tools as dt

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

TRAIN_IN = PROJECT_ROOT / "data" / "features" / "mordred_train.csv"
TEST_IN = PROJECT_ROOT / "data" / "features" / "mordred_test.csv"
TRAIN_OUT = PROJECT_ROOT / "data" / "features" / "filtered_train_pyqsar3.csv"
TEST_OUT = PROJECT_ROOT / "data" / "features" / "filtered_test_pyqsar3.csv"

ID_COLUMNS = ["SMILES", "logKoc"]
CORRELATION_THRESHOLD = 0.90

assert TRAIN_IN.exists(), f"Missing input file: {TRAIN_IN}"
assert TEST_IN.exists(), f"Missing input file: {TEST_IN}"


## Load Raw Mordred Descriptor Tables

The Phase 2 outputs are loaded as fixed inputs. `SMILES` and `logKoc` are preserved while descriptor columns are filtered strictly from Training-set decisions.

In [2]:
train_raw = pd.read_csv(TRAIN_IN)
test_raw = pd.read_csv(TEST_IN)

assert train_raw.shape == (514, 1615), f"Unexpected train shape: {train_raw.shape}"
assert test_raw.shape == (128, 1615), f"Unexpected test shape: {test_raw.shape}"
assert list(train_raw.columns) == list(test_raw.columns), "Train/Test raw schemas differ"
assert train_raw.columns[:2].tolist() == ID_COLUMNS, "Expected SMILES/logKoc as the first columns"

train_meta = train_raw[ID_COLUMNS].copy()
test_meta = test_raw[ID_COLUMNS].copy()
X_train_raw = train_raw.drop(columns=ID_COLUMNS).copy()
X_test_raw = test_raw.drop(columns=ID_COLUMNS).copy()

print(f"Raw Train shape: {train_raw.shape}")
print(f"Raw Test shape: {test_raw.shape}")
print(f"Raw descriptor count: {X_train_raw.shape[1]}")


Raw Train shape: (514, 1615)
Raw Test shape: (128, 1615)
Raw descriptor count: 1613


## Native PyQSAR3 Filtering Helpers

PyQSAR3's `FilteringTools` expects the first three columns to be `ID`, endpoint, and SMILES, with descriptors starting at column index 3. The helper below adapts the existing `SMILES`/`logKoc` tables to that native layout, calls the native methods, and returns the selected descriptor columns for application to both Train and Test sets.

In [3]:
def to_pyqsar_format(meta: pd.DataFrame, X: pd.DataFrame) -> pd.DataFrame:
    return pd.concat(
        [
            pd.Series(np.arange(1, len(meta) + 1), name="ID"),
            meta["logKoc"].rename("EP").reset_index(drop=True),
            meta["SMILES"].rename("SMI").reset_index(drop=True),
            X.reset_index(drop=True),
        ],
        axis=1,
    )


def make_filtering_tool(pyqsar_df: pd.DataFrame):
    tool = dt.FilteringTools.__new__(dt.FilteringTools)
    tool.X_data = pyqsar_df.copy()
    tool.X_data_ = pyqsar_df.copy()
    tool.nan_list = []
    tool.novar_list = []
    tool.inf_list = []
    tool.etc_list = []
    tool.etc_val_list = []
    return tool


def selected_descriptors_from_pyqsar_frame(pyqsar_df: pd.DataFrame) -> list[str]:
    return pyqsar_df.columns[3:].tolist()


## Native PyQSAR3 Non-Numeric, Missing, Infinite, and Zero-Variance Filters

The library-native `dt.NonNumricFilter` mirrors the project example and removes non-numeric/all-zero descriptors. `FilteringTools.rm_nan`, `rm_inf`, and `rm_novar` are then fitted on the Training set only. The resulting Training descriptor list is applied to the Test set unchanged.

In [4]:
# Native pyqsar3 example filter: numeric and non-all-zero descriptors.
X_train_numeric, numeric_descriptors = dt.NonNumricFilter(X_train_raw)
X_test_numeric = X_test_raw[numeric_descriptors].copy()

pyqsar_train_numeric = to_pyqsar_format(train_meta, X_train_numeric)
filter_tool = make_filtering_tool(pyqsar_train_numeric)

after_nan = filter_tool.rm_nan()
after_inf = filter_tool.rm_inf()
after_novar = filter_tool.rm_novar()

native_filtered_descriptors = selected_descriptors_from_pyqsar_frame(after_novar)
X_train_native = X_train_raw[native_filtered_descriptors].copy()
X_test_native = X_test_raw[native_filtered_descriptors].copy()

native_removed = X_train_raw.shape[1] - len(native_filtered_descriptors)

print(f"Descriptors after native PyQSAR3 filters: {len(native_filtered_descriptors)}")
print(f"Descriptors removed by native PyQSAR3 filters: {native_removed}")
print(f"Train native-filtered descriptor shape: {X_train_native.shape}")
print(f"Test native-filtered descriptor shape: {X_test_native.shape}")

assert X_train_native.shape[1] == X_test_native.shape[1], "Train/Test native-filtered feature counts differ"
assert list(X_train_native.columns) == list(X_test_native.columns), "Train/Test native-filtered schemas differ"
assert not X_train_native.isna().any().any(), "Training NaNs remain after native rm_nan"
assert np.isfinite(X_train_native.to_numpy(dtype=float)).all(), "Training Inf values remain after native rm_inf"


Start :  (514, 1613)
Filterd : (514, 1446)
NaN       : 491
Inf       : 0


No Var    : 0
Descriptors after native PyQSAR3 filters: 955
Descriptors removed by native PyQSAR3 filters: 658
Train native-filtered descriptor shape: (514, 955)
Test native-filtered descriptor shape: (128, 955)


## Train-Set Pearson High-Correlation Filter

The installed PyQSAR3 package provides correlation visualization utilities but no dedicated high-correlation pre-filter method. To complete the required pre-filtering while preserving the no-leakage rule, Pearson correlations are computed strictly on the Training descriptors that survived native PyQSAR3 filtering. The same dropped columns are then removed from the Test set.

In [5]:
corr_matrix = X_train_native.corr(method="pearson").abs()
upper_triangle = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
correlation_drop_columns = [
    column for column in upper_triangle.columns
    if (upper_triangle[column] > CORRELATION_THRESHOLD).any()
]

X_train_filtered = X_train_native.drop(columns=correlation_drop_columns)
X_test_filtered = X_test_native.drop(columns=correlation_drop_columns)

print(f"Correlation threshold: abs(r) > {CORRELATION_THRESHOLD}")
print(f"Highly correlated descriptors removed: {len(correlation_drop_columns)}")
print(f"Final descriptor count: {X_train_filtered.shape[1]}")

assert X_train_filtered.shape[1] == X_test_filtered.shape[1], "Train/Test final feature counts differ"
assert list(X_train_filtered.columns) == list(X_test_filtered.columns), "Train/Test final schemas differ"


Correlation threshold: abs(r) > 0.9
Highly correlated descriptors removed: 578
Final descriptor count: 377


## Save PyQSAR3-Filtered Outputs

The final outputs retain `SMILES`, `logKoc`, and the surviving descriptors. No Test-set statistics are used to fit any filter.

In [6]:
filtered_train = pd.concat(
    [train_meta.reset_index(drop=True), X_train_filtered.reset_index(drop=True)],
    axis=1,
)
filtered_test = pd.concat(
    [test_meta.reset_index(drop=True), X_test_filtered.reset_index(drop=True)],
    axis=1,
)

assert filtered_train.shape[0] == train_raw.shape[0], "Train row count changed"
assert filtered_test.shape[0] == test_raw.shape[0], "Test row count changed"
assert list(filtered_train.columns) == list(filtered_test.columns), "Filtered Train/Test schemas differ"

TRAIN_OUT.parent.mkdir(parents=True, exist_ok=True)
filtered_train.to_csv(TRAIN_OUT, index=False)
filtered_test.to_csv(TEST_OUT, index=False)

print(f"Saved Train output: {TRAIN_OUT}")
print(f"Saved Test output: {TEST_OUT}")
print(f"Final Train shape: {filtered_train.shape}")
print(f"Final Test shape: {filtered_test.shape}")


Saved Train output: /home/jun/Documents/qsar_modeling/data/features/filtered_train_pyqsar3.csv
Saved Test output: /home/jun/Documents/qsar_modeling/data/features/filtered_test_pyqsar3.csv
Final Train shape: (514, 379)
Final Test shape: (128, 379)
